### Threshold pcl5 >= 32



In [ ]:
from neo4j import GraphDatabase
import torch
from torch_geometric.data import HeteroData
from collections import defaultdict
import os
import random
import numpy as np
from datetime import datetime
import ast  # To safely convert string representations of lists to actual lists
import re
from langchain.chains import GraphCypherQAChain
from langchain_experimental.graph_transformers import LLMGraphTransformer
import torch.nn.functional as F
from torch_geometric.nn import (to_hetero, GraphConv, GATConv, GCNConv, SAGEConv, GATv2Conv, Linear, HeteroConv, HGTConv, RGCNConv, RGATConv, MessagePassing, global_add_pool)
from torch_geometric.loader import NeighborLoader
import torch_geometric.transforms as T
from torch_geometric.explain import GNNExplainer
import torch_geometric
import pyg_lib
import torch_sparse
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from langchain_core.documents import Document
from torch_geometric.utils import degree
import pandas as pd
from dotenv import load_dotenv

print("PyTorch Version:", torch.__version__)
print("PyG Loaded Successfully!")

PyTorch Version: 2.5.1+cu121
PyG Loaded Successfully!


In [2]:

# Load environment variables from .env file
load_dotenv()

# Set seed for reproducibility
def set_seed(seed_value=24):
    torch.manual_seed(seed_value)
    random.seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    np.random.seed(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(24)

In [ ]:
PTH = os.getenv("PTH")

x = "shiran"

# bartala
NEO4J_USERNAME = os.getenv("NEO4J_USER")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

NEO4J_URI = os.getenv("NEO4J_URI_"+x)
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD_"+x)

print(NEO4J_URI)

neo4j+s://ec508a85.databases.neo4j.io


In [ ]:
# Update Document nodes to contain pdi values.
# the goal is to make predictions for pcl5_total with pdi_1 to pdi_13 (no pdi_total).
# use SHAP analysis to indicate which of the pdi_x questions are the most important ones.
# can we remove some pdi questions or do we need all 13 questions (reflected in pdi_toal)?

# Load the CSV
df = pd.read_csv(os.path.join(PTH,"CBEx_pdi.csv"))

# Convert to list of dictionaries for Neo4j
data = df.to_dict(orient="records")

# Cypher query using UNWIND
query = """
UNWIND $data AS row
MATCH (d:Document)
WHERE d.id = row.record_id
SET d.cb_complication = row.cb_complication,
    d.pdi_q1 = row.pdi_q1,
    d.pdi_q2 = row.pdi_q2,
    d.pdi_q3 = row.pdi_q3,
    d.pdi_q4 = row.pdi_q4,
    d.pdi_q5 = row.pdi_q5,
    d.pdi_q6 = row.pdi_q6,
    d.pdi_q7 = row.pdi_q7,
    d.pdi_q8 = row.pdi_q8,
    d.pdi_q9 = row.pdi_q9,
    d.pdi_q10 = row.pdi_q10,
    d.pdi_q11 = row.pdi_q11,
    d.pdi_q12 = row.pdi_q12,
    d.pdi_q13 = row.pdi_q13
"""

# Run the update
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Run the update
def update_nodes(tx, data):
    tx.run(query, data=data)

with driver.session() as session:
    session.execute_write(update_nodes, data)

driver.close()

print(" Done updating nodes in Neo4j!")

 Done updating nodes in Neo4j!


# Read graph from neo4j

In [8]:
def load_all_nodes_and_rels_without_redundant_entity_label(driver) -> HeteroData:
    data = HeteroData()
    node_index_map = {}
    node_type_count = defaultdict(int)

    with driver.session() as session:
        node_query = """
        MATCH (n)
        RETURN elementId(n) AS nid, labels(n) AS labels, properties(n) AS props
        """
        node_records = session.run(node_query).data()

    nodeid_to_label_index = {}

    for record in node_records:
        nid = record["nid"]
        label_list = record["labels"]
        props = record["props"]

        # Remove unwanted labels and standardize naming
        label_list = [lbl for lbl in label_list if lbl not in {"__Entity__", "_Entity_"}]
        composite_label = "_".join(sorted(label_list)) if label_list else "Entity"
        composite_label = composite_label.replace("__", "_")

        if composite_label == "Unknown":
            continue  # Skip nodes with no valid label

        # Assign node index
        if (composite_label, nid) not in node_index_map:
            idx = node_type_count[composite_label]
            node_index_map[(composite_label, nid)] = idx
            node_type_count[composite_label] += 1

        nodeid_to_label_index[nid] = (composite_label, node_index_map[(composite_label, nid)])

        # Ensure all nodes have textEmbedding entry (even if None)
        if composite_label == "Document" and "textEmbedding" not in props:
            props["textEmbedding"] = None

        # Process node properties
        for prop_name, prop_val in props.items():
            if prop_name == "textEmbedding":
                if isinstance(prop_val, str):
                    try:
                        prop_val = ast.literal_eval(prop_val)
                        prop_val = torch.tensor(prop_val, dtype=torch.float) if isinstance(prop_val, list) else None
                    except:
                        prop_val = None
                elif isinstance(prop_val, list):
                    prop_val = torch.tensor(prop_val, dtype=torch.float)
                elif isinstance(prop_val, torch.Tensor):
                    pass  # already valid
                else:
                    prop_val = None
            elif isinstance(prop_val, (int, float)):
                prop_val = float(prop_val)

            if prop_name not in data[composite_label]:
                data[composite_label][prop_name] = [None] * node_type_count[composite_label]

            if len(data[composite_label][prop_name]) <= node_index_map[(composite_label, nid)]:
                data[composite_label][prop_name].extend(
                    [None] * (node_index_map[(composite_label, nid)] - len(data[composite_label][prop_name]) + 1))

            data[composite_label][prop_name][node_index_map[(composite_label, nid)]] = prop_val

        # Store node type mapping
        data[composite_label].setdefault("node_type", []).append(
            composite_label if composite_label == "Document" else props.get("id", "Unknown"))

    # Process Relationships (only original directed edges)
    with driver.session() as session:
        rel_query = """
        MATCH (s)-[r]->(t)
        RETURN elementId(r) AS rid, type(r) AS rtype, elementId(s) AS sid, elementId(t) AS tid, properties(r) AS props
        """
        rel_records = session.run(rel_query).data()

    for record in rel_records:
        sid = record["sid"]
        tid = record["tid"]
        rtype = record["rtype"]

        if sid not in nodeid_to_label_index or tid not in nodeid_to_label_index:
            continue

        s_label, s_idx = nodeid_to_label_index[sid]
        t_label, t_idx = nodeid_to_label_index[tid]

        edge_key = (s_label, rtype, t_label)
        edge_key = tuple(k.replace("__", "_") for k in edge_key)  # Clean double underscores

        if "edge_index" not in data[edge_key]:
            data[edge_key]["edge_index"] = [[], []]

        data[edge_key]["edge_index"][0].append(s_idx)
        data[edge_key]["edge_index"][1].append(t_idx)

    # Convert Edge Indices to Tensors
    for etype in data.edge_types:
        if "edge_index" in data[etype]:
            arr = data[etype]["edge_index"]
            if isinstance(arr, list):
                data[etype]["edge_index"] = torch.tensor(arr, dtype=torch.long)

    # Ensure `num_nodes` is explicitly set for all node types
    for ntype in data.node_types:
        if "num_nodes" not in data[ntype].__dict__ or data[ntype].num_nodes is None:
            if "id" in data[ntype]:
                data[ntype].num_nodes = len(data[ntype]["id"])
            elif "node_type" in data[ntype]:
                data[ntype].num_nodes = len(data[ntype]["node_type"])
            else:
                max_idx = -1
                for (src, _, tgt), edge_index in data.edge_index_dict.items():
                    if src == ntype and edge_index[0].numel() > 0:
                        max_idx = max(max_idx, edge_index[0].max().item())
                    if tgt == ntype and edge_index[1].numel() > 0:
                        max_idx = max(max_idx, edge_index[1].max().item())
                data[ntype].num_nodes = max_idx + 1 if max_idx >= 0 else 0

    print("Finished loading graph with explicit num_nodes and placeholder textEmbeddings.")
    return data

In [ ]:
# --- Run the function to fetch neo4j graph ----
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    connection_timeout=60  # optional
)

hetero_data = load_all_nodes_and_rels_without_redundant_entity_label(driver)
print(hetero_data)

In [10]:
# Compute and normalize the degree for each node type in a heterogeneous graph
def compute_node_degrees(hetero_data):
    """
    Compute and normalize the degree for each node type in a heterogeneous graph.
    - Store the raw degree in `deg` for 'Document' nodes.
    - Store the normalized degree in `x` for all other node types (as a 1D feature).
    """
    for ntype in hetero_data.node_types:
        num_nodes = hetero_data[ntype].num_nodes  # We assume num_nodes exists

        # Initialize degree tensor
        degrees = torch.zeros(num_nodes, dtype=torch.float)

        # Compute degree using `degree()` from torch_geometric.utils
        for (src, _, tgt), edge_index in hetero_data.edge_index_dict.items():
            if src == ntype:
                degrees += degree(edge_index[0], num_nodes, dtype=torch.float)
            if tgt == ntype:
                degrees += degree(edge_index[1], num_nodes, dtype=torch.float)

        # Normalize degrees (min-max scaling)
        if degrees.max() > 0:
            degrees = (degrees - degrees.min()) / (degrees.max() - degrees.min() + 1e-6)

        # Assign degree values
        if ntype == "Document":
            hetero_data[ntype]["deg"] = degrees.view(-1, 1)  # Store raw degree
        else:
            hetero_data[ntype]["x"] = degrees.view(-1, 1)  # Store normalized degree

    print("Finished computing and normalizing node degrees using PyG degree()!")

# Run the function
compute_node_degrees(hetero_data)

Finished computing and normalizing node degrees using PyG degree()!


In [ ]:
# Create the x tensor for Document nodes (textembeddign, degree, ...)

# Ensure textEmbedding is a stacked tensor
if isinstance(hetero_data['Document'].textEmbedding, list):
    hetero_data['Document'].textEmbedding = torch.stack([
        torch.tensor(v, dtype=torch.float) if not isinstance(v, torch.Tensor) else v
        for v in hetero_data['Document'].textEmbedding
    ])

# Ensure degree is also a tensor
degree = hetero_data['Document'].deg  # Should already be a tensor

cb_complication_tensor =  torch.tensor(hetero_data['Document'].cb_complication, dtype=torch.float32)
cb_complication_tensor = cb_complication_tensor.unsqueeze(1)

pdi_total_tensor = torch.tensor(hetero_data['Document'].pdi_total, dtype=torch.float32)
pdi_total_tensor = pdi_total_tensor.unsqueeze(1) # pdi is a 1D tensor, unsqueeze to make it 2D for concat

pdi_q1_tensor = torch.tensor(hetero_data['Document'].pdi_q1, dtype=torch.float32)
pdi_q2_tensor = torch.tensor(hetero_data['Document'].pdi_q2, dtype=torch.float32)
pdi_q3_tensor = torch.tensor(hetero_data['Document'].pdi_q3, dtype=torch.float32)
pdi_q4_tensor = torch.tensor(hetero_data['Document'].pdi_q4, dtype=torch.float32)
pdi_q5_tensor = torch.tensor(hetero_data['Document'].pdi_q5, dtype=torch.float32)
pdi_q6_tensor = torch.tensor(hetero_data['Document'].pdi_q6, dtype=torch.float32)
pdi_q7_tensor = torch.tensor(hetero_data['Document'].pdi_q7, dtype=torch.float32)
pdi_q8_tensor = torch.tensor(hetero_data['Document'].pdi_q8, dtype=torch.float32)
pdi_q9_tensor = torch.tensor(hetero_data['Document'].pdi_q9, dtype=torch.float32)
pdi_q10_tensor = torch.tensor(hetero_data['Document'].pdi_q10, dtype=torch.float32)
pdi_q11_tensor = torch.tensor(hetero_data['Document'].pdi_q11, dtype=torch.float32)
pdi_q12_tensor = torch.tensor(hetero_data['Document'].pdi_q12, dtype=torch.float32)
pdi_q13_tensor = torch.tensor(hetero_data['Document'].pdi_q13, dtype=torch.float32)


# unsqueeze to make it 2D for concat
pdi_q1_tensor = pdi_q1_tensor.unsqueeze(1)
pdi_q2_tensor = pdi_q2_tensor.unsqueeze(1)
pdi_q3_tensor = pdi_q3_tensor.unsqueeze(1)
pdi_q4_tensor = pdi_q4_tensor.unsqueeze(1)
pdi_q5_tensor = pdi_q5_tensor.unsqueeze(1)
pdi_q6_tensor = pdi_q6_tensor.unsqueeze(1)
pdi_q7_tensor = pdi_q7_tensor.unsqueeze(1)
pdi_q8_tensor = pdi_q8_tensor.unsqueeze(1)
pdi_q9_tensor = pdi_q9_tensor.unsqueeze(1)
pdi_q10_tensor = pdi_q10_tensor.unsqueeze(1)
pdi_q11_tensor = pdi_q11_tensor.unsqueeze(1)
pdi_q12_tensor = pdi_q12_tensor.unsqueeze(1)
pdi_q13_tensor = pdi_q13_tensor.unsqueeze(1)

Now create the `x` matrix for Document nodes

In [27]:
# Concatenate the degree and textEmbedding
hetero_data['Document'].x = torch.cat([
    degree,
    hetero_data['Document'].textEmbedding,
    cb_complication_tensor,
    pdi_total_tensor,
    pdi_q1_tensor,
    pdi_q2_tensor,
    pdi_q3_tensor,
    pdi_q4_tensor,
    pdi_q5_tensor,
    pdi_q6_tensor,
    pdi_q7_tensor,
    pdi_q8_tensor,
    pdi_q9_tensor,
    pdi_q10_tensor,
    pdi_q11_tensor,
    pdi_q12_tensor,
    pdi_q13_tensor
], dim=1)

In [ ]:
# Define the y attribute for Document nodes based on 'ptsd'
if 'ptsd' in hetero_data['Document']:
    ptsd_vals = hetero_data['Document']['ptsd']

    # Convert to tensor if needed
    if isinstance(ptsd_vals, list):
        ptsd_vals = [
            0.0 if v is None else float(v)
            for v in ptsd_vals
        ]
        ptsd_vals = torch.tensor(ptsd_vals, dtype=torch.float)

    elif isinstance(ptsd_vals, torch.Tensor):
        ptsd_vals = ptsd_vals.clone().detach().float()

    hetero_data['Document'].y = ptsd_vals
else:
    print("'ptsd' attribute not found in Document nodes.")

In [ ]:
# Function to sanitize node/edge types
def clean_type_name(type_name):
    type_name = re.sub(r'[^a-zA-Z0-9_]', '_', type_name)
    type_name = re.sub(r'_+', '_', type_name)
    return type_name.strip('_')

# Original to sanitized mapping and new indices
node_type_map = {}
node_new_index = {}

fixed_data = HeteroData()

# Step 1: Re-index nodes while preserving unique identities
for node_type in hetero_data.node_types:
    clean_node_type = clean_type_name(node_type)

    # Initialize sanitized node type storage if not present
    if clean_node_type not in fixed_data.node_types:
        fixed_data[clean_node_type].num_nodes = 0
        node_new_index[clean_node_type] = 0

    num_old_nodes = hetero_data[node_type].num_nodes
    node_type_map[(node_type)] = (clean_node_type, node_new_index[clean_node_type])

    # Copy all node attributes and adjust index mapping
    for attr, values in hetero_data[node_type].items():
        if isinstance(values, torch.Tensor):
            if attr not in fixed_data[clean_node_type]:
                fixed_data[clean_node_type][attr] = values.clone()
            else:
                fixed_data[clean_node_type][attr] = torch.cat(
                    [fixed_data[clean_node_type][attr], values], dim=0
                )
        else:
            # If it's not a tensor, store as list and append
            if attr not in fixed_data[clean_node_type]:
                # Initialize as list
                if isinstance(values, list):
                    fixed_data[clean_node_type][attr] = values.copy()
                else:
                    fixed_data[clean_node_type][attr] = [values]
            else:
                if isinstance(values, list):
                    fixed_data[clean_node_type][attr].extend(values)
                else:
                    # Convert the current scalar value to a list (only once)
                    if not isinstance(fixed_data[clean_node_type][attr], list):
                        fixed_data[clean_node_type][attr] = [fixed_data[clean_node_type][attr]]
                    fixed_data[clean_node_type][attr].append(values)

    # Update new indices for next set of nodes
    node_new_index[clean_node_type] += num_old_nodes
    fixed_data[clean_node_type].num_nodes = node_new_index[clean_node_type]

# Step 2: Adjust edge indices according to new node indexing
for (src, rel, dst), edge_store in hetero_data.edge_items():
    clean_src, src_offset = node_type_map[src]
    clean_dst, dst_offset = node_type_map[dst]
    clean_rel = clean_type_name(rel)

    edge_index = edge_store.edge_index.clone()
    edge_index[0] += src_offset
    edge_index[1] += dst_offset

    # Merge edges into fixed_data
    if 'edge_index' not in fixed_data[(clean_src, clean_rel, clean_dst)]:
        fixed_data[(clean_src, clean_rel, clean_dst)].edge_index = edge_index
    else:
        fixed_data[(clean_src, clean_rel, clean_dst)].edge_index = torch.cat([
            fixed_data[(clean_src, clean_rel, clean_dst)].edge_index,
            edge_index
        ], dim=1)

# Replace original data with sanitized data
hetero_data = fixed_data

print("Finished correct sanitization with proper re-indexing.")


In [ ]:
torch.save(hetero_data, os.path.join(PTH,"hetero_graph_attribute.pt"))
print("Graph saved successfully.")

# GNN

In [4]:
# load the graph
hetero_data = torch.load(os.path.join(PTH,"hetero_graph_attribute.pt"), weights_only=False)
print("Graph loaded successfully.")

Graph loaded successfully.


In [5]:
# make sure there is no node with indegree 0 (to enable learning with GraphSAGE)
from copy import deepcopy

# Identify all destination node types
dst_node_types = set([dst for (_, _, dst) in hetero_data.edge_types])

# For edge types where the source node type is not in dst_node_types, add reverse edges
new_edges = {}
for (src, rel, dst), edge_index in hetero_data.edge_index_dict.items():
    if src not in dst_node_types:
        rev_edge_type = (dst, f"{rel}_rev", src)
        rev_edge_index = edge_index.flip(0)  # Swap rows to reverse direction
        if rev_edge_type not in hetero_data.edge_types:
            new_edges[rev_edge_type] = rev_edge_index

# Add the missing reverse edges
for rev_edge_type, rev_edge_index in new_edges.items():
    hetero_data[rev_edge_type].edge_index = rev_edge_index

In [6]:
# check that all nodes have at least one incomming link

destination_types = set(tgt for _, _, tgt in hetero_data.edge_types)

for ntype in hetero_data.node_types:
    if ntype not in destination_types:
        print(f" Node type '{ntype}' is not a destination in any edge type!")

In [7]:
# Splitting 'Document' nodes into train/test/val splits
doc_indices = torch.arange(hetero_data['Document'].num_nodes)
y_labels = hetero_data['Document'].y

train_idx, test_idx = train_test_split(
    doc_indices, stratify=y_labels, test_size=0.1, random_state=42
)
train_idx, val_idx = train_test_split(
    train_idx, stratify=y_labels[train_idx], test_size=0.1, random_state=42
)

# initialize a boolean tensor of the same size as y_labels, filled entirely with False
hetero_data['Document'].train_mask = torch.zeros_like(y_labels, dtype=torch.bool)
hetero_data['Document'].val_mask = torch.zeros_like(y_labels, dtype=torch.bool)
hetero_data['Document'].test_mask = torch.zeros_like(y_labels, dtype=torch.bool)

# now set the mask id by seting True to spesific indices
hetero_data['Document'].train_mask[train_idx] = True
hetero_data['Document'].val_mask[val_idx] = True
hetero_data['Document'].test_mask[test_idx] = True

In [ ]:
y_labels = hetero_data['Document'].y
test_mask = hetero_data['Document'].test_mask

# Get only the y values for the test set
y_test = y_labels[test_mask]

# Count how many of each class
num_class_0 = (y_test == 0).sum().item()
num_class_1 = (y_test == 1).sum().item()

print(f"Test set examples — Class 0: {num_class_0}, Class 1: {num_class_1}")

# GNN Training

In [9]:
# Define a GraphSAGE model
class GNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

In [10]:
# Instantiate and convert model to heterogeneous form
model = GNN(hidden_channels=64, out_channels=2)
model = to_hetero(model, hetero_data.metadata(), aggr='mean')

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

In [10]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# Training loop
def train():
    model.train()
    optimizer.zero_grad()
    out = model(hetero_data.x_dict, hetero_data.edge_index_dict)['Document']
    loss = F.cross_entropy(out[hetero_data['Document'].train_mask], hetero_data['Document'].y[hetero_data['Document'].train_mask].long())
    print(f"Loss: {loss.item()}")
    
    if torch.isnan(loss):
        raise ValueError("NaN loss detected!")
    
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(mask,flag):
    model.eval()
    out = model(hetero_data.x_dict, hetero_data.edge_index_dict)['Document']
    pred = out[mask].argmax(dim=1)
    true_labels = hetero_data['Document'].y[mask]
    acc = (pred == hetero_data['Document'].y[mask]).sum().item() / mask.sum().item()
    
    # Compute F1-score
    y_true = true_labels.cpu()
    y_pred = pred.cpu()
    f1 = f1_score(y_true, y_pred, average="macro")  # or "macro"

    if flag:
        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        print("Confusion Matrix:\n", cm)

        # Optional: full classification report
        print("\n Classification Report:")
        print(classification_report(y_true, y_pred))
    return acc, f1

In [ ]:
# Run training
for epoch in range(1, 101):
    loss = train()
    train_acc, train_f1 = evaluate(hetero_data['Document'].train_mask, flag=False)
    val_acc, val_f1 = evaluate(hetero_data['Document'].val_mask, flag=False)
    if epoch % 10 == 0:
        print(f'Epoch {epoch:03d}, Loss: {loss:.4f}, '
              f'Train Acc: {train_acc:.4f}, Train F1: {train_f1:.4f}, '
              f'Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}')

In [57]:
# Save the trained heterogeneous model after training is done
torch.save(model.state_dict(), os.path.join(PTH,'gnn_trained_hetero_model.pth'))
print("Model saved to: ",  os.path.join(PTH,'gnn_trained_hetero_model.pth'))

Model saved to:  /home/bartalab/github/CBPTSD_GRAPHRAG/data/gnn_trained_hetero_model.pth


In [ ]:
# Final test evaluation
test_acc, test_f1 = evaluate(hetero_data['Document'].test_mask, flag = True)
print(f'Test Accuracy: {test_acc:.4f}, Test F1-score: {test_f1:.4f}')

# Ablation study

Help functions

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

hetero_data = hetero_data.to(device)


# === Function to rebuild Document.x with selected features to exclude ===
def build_document_x(exclude: list[str] = []) -> torch.Tensor:
    doc = hetero_data['Document']
    device = doc.textEmbedding.device
    features = []
    featurs_included = ['degree'] 

    if exclude is None:
        exclude = []

    # Degree is always included
    features.append(doc.deg.to(device))

    # embeddings
    if 'textEmbedding' not in exclude:
       features.append(doc.textEmbedding.to(device))
       featurs_included.append('textEmbedding')

    # cb_complication
    if 'cb_complication' not in exclude:
            raw_feat = getattr(doc, 'cb_complication')
            # If it's a list, convert to tensor
            if isinstance(raw_feat, list):
                raw_feat = torch.tensor(raw_feat, dtype=torch.float32)
            # Ensure 2D shape [num_nodes, 1]
            if raw_feat.dim() == 1:
                raw_feat = raw_feat.unsqueeze(1)
            features.append(raw_feat.to(device))
            featurs_included.append('cb_complication')

    # PDI questions
    for i in range(1, 14):
        key = f'pdi_q{i}'
        if key not in exclude: # if key not in the exclude list --> add it to the features list (X matrix)
            featurs_included.append(key)
            raw_feat = getattr(doc, key)
            # If it's a list, convert to tensor
            if isinstance(raw_feat, list):
                raw_feat = torch.tensor(raw_feat, dtype=torch.float32)
            # Ensure 2D shape [num_nodes, 1]
            if raw_feat.dim() == 1:
                raw_feat = raw_feat.unsqueeze(1)
            features.append(raw_feat.to(device))
            
    print(featurs_included)
    return torch.cat(features, dim=1)


# === Function to reset the model before each run ===
def reset_model():
    # Instantiate and convert model to heterogeneous form
    model = GNN(hidden_channels=64, out_channels=2)
    model = to_hetero(model, hetero_data.metadata(), aggr='mean')
    return model.to(device)

### Manually drop attributes 

In [268]:

feature_to_drop = [
                #'cb_complication',
                'textEmbedding', 
                'pdi_q1',
                #'pdi_q2',
                #'pdi_q3',
                'pdi_q4',
                #'pdi_q5',
                # 'pdi_q6',
                #'pdi_q7',
                'pdi_q8',
                'pdi_q9',
                'pdi_q10',
                #  'pdi_q11',
                #  'pdi_q12',
                'pdi_q13',
                    ]

# Rebuild Document.x with selected features
hetero_data['Document'].x = build_document_x(exclude=feature_to_drop)
print(hetero_data['Document'].x.device)  # should be cuda:0


# Reset model and optimizer
model = reset_model()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

# Train the model
for epoch in range(1,11):
    loss = train()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}")

# Final test evaluation
test_acc, test_f1 = evaluate(hetero_data['Document'].test_mask, flag=True)
print(f"Test Accuracy: {test_acc:.4f}, Test F1-score: {test_f1:.4f}")

False
['degree', 'cb_complication', 'pdi_q2', 'pdi_q3', 'pdi_q5', 'pdi_q6', 'pdi_q7', 'pdi_q11', 'pdi_q12']
cuda:0
Loss: 0.6909400224685669
Loss: 0.6874459981918335
Loss: 0.6839918494224548
Loss: 0.6802775859832764
Loss: 0.6760756969451904
Loss: 0.6712833642959595
Loss: 0.6658691763877869
Loss: 0.6597239971160889
Loss: 0.6528521180152893
Loss: 0.6454131007194519
Epoch 010, Loss: 0.6454
Confusion Matrix:
 [[19  1]
 [ 1  9]]

 Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      0.95      0.95        20
         1.0       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

Test Accuracy: 0.9333, Test F1-score: 0.9250


### Auto feature importance

In [ ]:
# === Backwards stepwise regression --> Begin ablation loop ===

all_features = ['pdi_q1', 'pdi_q2', 'pdi_q3', 'pdi_q4', 'pdi_q5', 'pdi_q6', 'pdi_q7', 'pdi_q8', 'pdi_q9', 'pdi_q10', 'pdi_q11', 'pdi_q12', 'pdi_q13','cb_complication','textEmbedding']
ablation_results = []

for feature_to_drop in all_features:
    print(f"\n ======= Dropping feature: {feature_to_drop} ========")

    # Rebuild feature matrix without selected feature
    hetero_data['Document'].x = build_document_x(exclude=[feature_to_drop])
    print(hetero_data['Document'].x.device)  # should be cuda:0

    # Reset model and optimizer
    model = reset_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

    # Train the model
    for epoch in range(1, 11):
        loss = train()
        if epoch % 10 == 0:
            print(f"Epoch {epoch:03d}, Loss: {loss:.4f}")

    # Final test evaluation
    test_acc, test_f1 = evaluate(hetero_data['Document'].test_mask, flag=True)
    print(f"Test Accuracy: {test_acc:.4f}, Test F1-score: {test_f1:.4f}")

    ablation_results.append({
        'excluded_feature': feature_to_drop,
        'test_f1_score': test_f1
    })


# === Display and save results ===
df = pd.DataFrame(ablation_results)
print("\n Test F1-score after dropping each feature:\n")
print(df.to_string(index=False))

# Optionally save
df.to_csv("ablation_test_f1_results_pdi.csv", index=False)